# M4 저장 상태 복원 오류 진단 및 재개

기존 실행에서 상태를 불러온 결과가 원래 상태와 달라 멈췄습니다. 이 노트북은 완료된 상태를 보존하고, **실패했던 상황부터 어떤 차이가 났는지 확인**합니다.

1. 오류가 난 기존 노트북이 중단된 상태인지 확인합니다.
2. 이 노트북을 CPU 런타임으로 열고 **런타임 → 모두 실행**을 누릅니다.
3. 같은 문제가 다시 나면 신호·관측·차량 상태 차이를 Drive에 기록하고 멈춥니다. 저에게 오류가 났다고 알려 주세요.
4. 해당 상황이 통과하면 남은 진단을 자동으로 이어갑니다. 끝나면 알려 주세요.

요청대로 작업시간이나 계산량 한도로 자동 중단하지 않습니다. 원래의 복원 검사와 연구 조건은 유지하며, 연결이 끊기면 다시 실행해 이어갈 수 있습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE_ROOT / "FutureTraffic_M4PROBE_RESTORE.zip"
SOURCE = DRIVE_ROOT / "explore_weight_v1"
FIXED = DRIVE_ROOT / "explore_fixed24_v1"
NIGHT = DRIVE_ROOT / "explore_night_v1"
OUT = DRIVE_ROOT / "explore_m4probe_v1"
WORKSPACE = Path("/content/FutureTraffic_M4PROBE_RESTORE")
WORK = Path("/content/m4probe_restore_work")
for path in (ARCHIVE, SOURCE / "experiment.json", FIXED / "experiment.json", FIXED / "budget.json", NIGHT / "experiment.json", OUT / "experiment.json"):
    assert path.is_file(), f"필요한 파일이 없습니다: {path}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 기존 결과를 이어갈 폴더:", OUT)

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib
import json
import stat
import sys
import types
import zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "M4_PROBE_RESTORE_V1", "M4 복원 진단 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델은 Drive 원본에서 읽어야 합니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("m4probe_restore_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE}")

In [ ]:
import subprocess
import importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}
source_packages = json.loads((SOURCE / "experiment.json").read_text())["packages"]
assert all(source_packages.get(name) == version for name, version in EXPECTED.items()), "출발 실험의 패키지 버전이 이 묶음과 다릅니다."

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("출발 실험과 패키지 버전이 모두 같아 재설치를 건너뜁니다.")
else:
    print("출발 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 출발 실험과 다릅니다: {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
import subprocess
import signal
from IPython.display import display, Markdown
ENV = {**os.environ, "FOCUSED_RULESET": "pet1", "FOCUSED_NORTH_EXT": "240", "PYTHONUNBUFFERED": "1",
       "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
def run_live(args):
    process = subprocess.Popen([sys.executable, "-u", *args], cwd=WORKSPACE, env=ENV,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
    try:
        for line in process.stdout:
            print(line.rstrip())
        code = process.wait()
    except KeyboardInterrupt:
        try:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait(timeout=20)
        except ProcessLookupError:
            pass
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
        raise
    if code != 0:
        raise RuntimeError(f"작업이 중단됐습니다 (종료 코드 {code}). 위의 원인을 확인해 주세요.")

for pattern in ("test_explore_m4probe_continue.py", "test_explore_m4probe_diagnostics.py"):
    run_live(["-m", "unittest", "discover", "-s", "tests", "-p", pattern])
print("실행 전 검사 통과")

In [ ]:
# 기존 완료 결과를 검증해 재사용하고, 실패했던 상황부터 재검사합니다.
try:
    run_live(["src/explore_m4probe_continue.py", "--out", str(OUT), "--work", str(WORK),
              "--source", str(SOURCE), "--fixed", str(FIXED), "--night", str(NIGHT), "--workers", "0"])
finally:
    summary = OUT / "summary.md"
    if summary.is_file():
        display(Markdown(summary.read_text()))
    print("진단 파일 폴더:", OUT / "restore_diagnostics")
    print("결과 폴더:", OUT)